In [ ]:
from pathlib import Path
import os, stat, glob, traceback
import numpy as np
import pandas as pd
import arviz as az
from cmdstanpy import CmdStanModel
import matplotlib.pyplot as plt

/Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [ ]:
os.environ.pop('CXX', None)
os.environ.pop('CC', None)

In [ ]:
# Paths and helper to compile models
stan_dir = Path('../stan').resolve()
out_root = Path.cwd() / 'stan_output'
out_root.mkdir(parents=True, exist_ok=True)
models = {
    'minimal': {'file': stan_dir / 'minimal.stan', 'outdir': out_root / 'minimal'},
    'province': {'file': stan_dir / 'logistic_province.stan', 'outdir': out_root / 'province'},
    'full': {'file': stan_dir / 'logistic_full.stan', 'outdir': out_root / 'full'},
    'response': {'file': stan_dir / 'logistic_response.stan', 'outdir': out_root / 'response'},
}
for m in models.values():
    m['outdir'].mkdir(parents=True, exist_ok=True)

def compile_model(stan_path):
    print('Compiling', stan_path.name)
    mdl = CmdStanModel(stan_file=str(stan_path))
    mode = os.stat(mdl.exe_file).st_mode
    os.chmod(mdl.exe_file, mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    return mdl

### Data Prep

In [ ]:
# Load data and prepare stan_data objects (one-off)
df = pd.read_csv('../data/fire_db.csv')
# minimal
df0 = df[['any_evacuation', 'log_fire_size_ha', 'fn_indicator']].dropna().copy()
stan_data_0 = {
    'N': len(df0),
    'y': df0['any_evacuation'].astype(int).tolist(),
    'log_fire_size': df0['log_fire_size_ha'].astype(float).tolist(),
    'fn_indicator': df0['fn_indicator'].astype(int).tolist(),
}
# province
df_proc = df.dropna(subset=['any_evacuation','log_fire_size_ha','fn_indicator','province']).copy()
prov_cat = pd.Categorical(df_proc['province'])
df_proc['prov_idx'] = prov_cat.codes + 1
stan_data_province = {
    'N': len(df_proc),
    'P': len(prov_cat.categories),
    'province': df_proc['prov_idx'].astype(int).tolist(),
    'y': df_proc['any_evacuation'].astype(int).tolist(),
    'log_fire_size': df_proc['log_fire_size_ha'].astype(float).tolist(),
    'fn_indicator': df_proc['fn_indicator'].astype(int).tolist(),
}
# full
df_full = df.dropna(subset=['any_evacuation','log_fire_size_ha','fn_indicator','log_dist_to_fn_km','n_fn_20km','fire_cause','fire_type','protection_zone','province']).copy()
prov_cat_full = pd.Categorical(df_full['province'])
cause_cat = pd.Categorical(df_full['fire_cause'])
type_cat = pd.Categorical(df_full['fire_type'])
prot_cat = pd.Categorical(df_full['protection_zone'])

_log_fire = df_full['log_fire_size_ha'].astype(float).to_numpy()
_dist = df_full['log_dist_to_fn_km'].astype(float).to_numpy()
# _nfn = df_full['n_fn_20km'].astype(float).to_numpy()
stan_data_full = {
    'N': len(df_full),
    'P': len(prov_cat_full.categories),
    'province': (prov_cat_full.codes + 1).tolist(),
    'y': df_full['any_evacuation'].astype(int).tolist(),
    'log_fire_size': _log_fire.tolist(),
    'log_dist_to_fn_km': _dist.tolist(),
    # 'n_fn_20km': _nfn.tolist(),
    'fn_indicator': df_full['fn_indicator'].astype(int).tolist(),
    # 'K_cause': len(cause_cat.categories),
    # 'fire_cause': (cause_cat.codes + 1).tolist(),
    # 'K_type': len(type_cat.categories),
    # 'fire_type': (type_cat.codes + 1).tolist(),
    'K_prot': len(prot_cat.categories),
    'protection_zone': (prot_cat.codes + 1).tolist(),
}
# response
df_resp = df.dropna(subset=['any_evacuation','log_fire_size_ha','fn_indicator','log_dist_to_fn_km','n_fn_20km','fire_cause','fire_type','protection_zone','response_type','province']).copy()
prov_cat_r = pd.Categorical(df_resp['province'])
cause_cat_r = pd.Categorical(df_resp['fire_cause'])
type_cat_r = pd.Categorical(df_resp['fire_type'])
resp_cat = pd.Categorical(df_resp['response_type'])
prot_cat_r = pd.Categorical(df_resp['protection_zone'])
stan_data_response = {
    'N': len(df_resp),
    'P': len(prov_cat_r.categories),
    'province': (prov_cat_r.codes + 1).tolist(),
    'y': df_resp['any_evacuation'].astype(int).tolist(),
    'log_fire_size': df_resp['log_fire_size_ha'].astype(float).tolist(),
    'log_dist_to_fn_km': df_resp['log_dist_to_fn_km'].astype(float).tolist(),
    # 'n_fn_20km': df_resp['n_fn_20km'].astype(int).tolist(),
    'fn_indicator': df_resp['fn_indicator'].astype(int).tolist(),
    # 'K_cause': len(cause_cat_r.categories),
    # 'fire_cause': (cause_cat_r.codes + 1).tolist(),
    # 'K_type': len(type_cat_r.categories),
    # 'fire_type': (type_cat_r.codes + 1).tolist(),
    'K_response': len(resp_cat.categories),
    'response_type': (resp_cat.codes + 1).tolist(),
    'K_prot': len(prot_cat_r.categories),
    'protection_zone': (prot_cat_r.codes + 1).tolist(),
}
print('Prepared data sizes: minimal', stan_data_0['N'], 'province', stan_data_province['N'], 'full', stan_data_full['N'], 'response', stan_data_response['N'])

Prepared data sizes: minimal 15203 province 15203 full 15203 response 15203


In [ ]:
# Helper to run, convert and save InferenceData for a single model
def run_and_save(name, mdl, data, outdir, chains=4, warmup=500, sampling=500, adapt_delta=0.95, show_console=False):
    print('Running', name)
    fit = mdl.sample(
        data=data,
        chains=chains,
        parallel_chains=min(chains, os.cpu_count() or 1),
        iter_warmup=warmup,
        iter_sampling=sampling,
        seed=20260417,
        show_console=show_console,
        adapt_delta=adapt_delta,
        output_dir=str(outdir),
        save_warmup=False,
    )
    idata = az.from_cmdstanpy(posterior=fit, posterior_predictive='y_rep', log_likelihood='log_lik')
    fn = outdir / 'idata.nc'
    idata.to_netcdf(str(fn))
    print('Saved idata to', fn)
    return fit, idata

## Fit Models

In [ ]:
chains = 4
warmup = 500
sampling = 1000
adapt_delta = 0.95

In [ ]:
mdl_min = compile_model(models['minimal']['file'])
fit_min, idata_min = run_and_save('minimal', mdl_min, stan_data_0, models['minimal']['outdir'], chains=chains, warmup=warmup, sampling=sampling, adapt_delta=adapt_delta)

Compiling minimal.stan
Running minimal


09:30:26 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/1500 [00:00<?, ?it/s, (Warmup)]





chain 1:   7%|▋         | 100/1500 [00:02<00:37, 36.94it/s, (Warmup)]


chain 1:  13%|█▎        | 200/1500 [00:04<00:29, 44.74it/s, (Warmup)]


chain 1:  20%|██        | 300/1500 [00:05<00:21, 55.95it/s, (Warmup)]


chain 1:  27%|██▋       | 400/1500 [00:06<00:16, 67.24it/s, (Warmup)]





chain 1:  33%|███▎      | 500/1500 [00:08<00:13, 73.33it/s, (Sampling)]


chain 1:  40%|████      | 600/1500 [00:11<00:18, 49.95it/s, (Sampling)]


chain 1:  47%|████▋     | 700/1500 [00:13<00:17, 45.50it/s, (Sampling)]


chain 1:  53%|█████▎    | 800/1500 [00:16<00:16, 41.91it/s, (Sampling)]


chain 1:  60%|██████    | 900/1500 [00:19<00:15, 38.94it/s, (Sampling)]


chain 1:  67%|██████▋   | 1000/1500 [00:22<00:13, 35.98it/s, (Sampling)]


chain 1:  73%|███████▎  | 1100/1500 [00:25<00:11, 36.17it/s, (Sampling)]


chain 1:  80%|████████  | 1200/1500 [00:28<00:08, 36.97it/s, (Samplin


09:31:05 - cmdstanpy - INFO - CmdStan done processing.



Saved idata to /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/minimal/idata.nc


In [ ]:
mdl_prov = compile_model(models['province']['file'])
fit_prov, idata_prov = run_and_save('province', mdl_prov, stan_data_province, models['province']['outdir'], chains=chains, warmup=warmup, sampling=sampling, adapt_delta=adapt_delta)

Compiling logistic_province.stan
Running province


09:31:27 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/1500 [00:00<?, ?it/s, (Warmup)]



chain 1:   7%|▋         | 100/1500 [00:16<03:44,  6.24it/s, (Warmup)]


chain 1:  13%|█▎        | 200/1500 [00:30<03:17,  6.57it/s, (Warmup)]


chain 1:  20%|██        | 300/1500 [00:41<02:40,  7.46it/s, (Warmup)]


chain 1:  27%|██▋       | 400/1500 [00:53<02:17,  8.01it/s, (Warmup)]


chain 1:  33%|███▎      | 500/1500 [01:04<01:59,  8.37it/s, (Sampling)]





chain 1:  47%|████▋     | 700/1500 [01:19<01:16, 10.40it/s, (Sampling)]


chain 1:  53%|█████▎    | 800/1500 [01:28<01:05, 10.61it/s, (Sampling)]


chain 1:  67%|██████▋   | 1000/1500 [01:48<00:48, 10.26it/s, (Sampling)]


chain 1:  73%|███████▎  | 1100/1500 [01:57<00:38, 10.48it/s, (Sampling)]


chain 1:  87%|████████▋ | 1300/1500 [02:14<00:17, 11.30it/s, (Sampling)]


chain 1: 100%|██████████| 1500/1500 [02:29<00:00, 12.00it/s, (Sampling)]







chain 1: 100%|██████████| 1500/1500 [02:49<00:00, 12.00it/s, (Sa


09:34:31 - cmdstanpy - INFO - CmdStan done processing.



Saved idata to /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/province/idata.nc


In [ ]:
mdl_full = compile_model(models['full']['file'])
fit_full, idata_full = run_and_save('full', mdl_full, stan_data_full, models['full']['outdir'], chains=chains, warmup=warmup, sampling=sampling, adapt_delta=adapt_delta)

09:36:15 - cmdstanpy - INFO - compiling stan file /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/stan/logistic_full.stan to exe file /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/stan/logistic_full


Compiling logistic_full.stan


09:36:25 - cmdstanpy - INFO - compiled model executable: /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/stan/logistic_full


Running full


09:36:25 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/1500 [00:00<?, ?it/s, (Warmup)]





chain 1:   7%|▋         | 100/1500 [00:44<10:24,  2.24it/s, (Warmup)]


chain 1:  13%|█▎        | 200/1500 [01:23<08:54,  2.43it/s, (Warmup)]

chain 1:  20%|██        | 300/1500 [01:57<07:36,  2.63it/s, (Warmup)]



chain 1:  27%|██▋       | 400/1500 [02:28<06:28,  2.83it/s, (Warmup)]










chain 1:  33%|███▎      | 501/1500 [03:06<05:58,  2.79it/s, (Sampling)]

chain 1:  40%|████      | 600/1500 [03:33<04:51,  3.08it/s, (Sampling)]


chain 1:  47%|████▋     | 700/1500 [03:55<03:47,  3.52it/s, (Sampling)]


chain 1:  53%|█████▎    | 800/1500 [04:26<03:26,  3.40it/s, (Sampling)]

chain 1:  60%|██████    | 900/1500 [04:57<02:59,  3.34it/s, (Sampling)]


chain 1:  67%|██████▋   | 1000/1500 [05:23<02:22,  3.50it/s, (Sampling)]


chain 1:  73%|███████▎  | 1100/1500 [05:50<01:52,  3.54it/s, (Sampling)]

chain 1:  80%|████████  | 1200/1500 [06:20<01:26,  3.47it/s, (Sampl


09:44:49 - cmdstanpy - INFO - CmdStan done processing.



Saved idata to /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/full/idata.nc


## Response Model

In [ ]:
# Build response-model data to match logistic_response.stan
req_cols_resp = [
    'any_evacuation',
    'log_fire_size_ha',
    'log_dist_to_fn_km',
    'fn_indicator',
    'province',
    'response_type',
    'protection_zone',
]

df_resp = df.dropna(subset=req_cols_resp).copy()

prov_cat_r = pd.Categorical(df_resp['province'])
resp_cat = pd.Categorical(df_resp['response_type'])
prot_cat_r = pd.Categorical(df_resp['protection_zone'])

stan_data_response = {
    'N': len(df_resp),
    'P': len(prov_cat_r.categories),
    'province': (prov_cat_r.codes + 1).astype(int).tolist(),
    'y': df_resp['any_evacuation'].astype(int).tolist(),
    'log_fire_size': df_resp['log_fire_size_ha'].astype(float).tolist(),
    'log_dist_to_fn_km': df_resp['log_dist_to_fn_km'].astype(float).tolist(),
    'fn_indicator': df_resp['fn_indicator'].astype(int).tolist(),
    'K_response': len(resp_cat.categories),
    'response_type': (resp_cat.codes + 1).astype(int).tolist(),
    'K_prot': len(prot_cat_r.categories),
    'protection_zone': (prot_cat_r.codes + 1).astype(int).tolist(),
}

print(
    'Prepared response data:',
    'N =', stan_data_response['N'],
    '| P =', stan_data_response['P'],
    '| K_response =', stan_data_response['K_response'],
    '| K_prot =', stan_data_response['K_prot'],
)

Prepared response data: N = 15203 | P = 12 | K_response = 4 | K_prot = 8


In [ ]:
mdl_response = compile_model(models['response']['file'])
fit_response, idata_response = run_and_save(
    'response',
    mdl_response,
    stan_data_response,
    models['response']['outdir'],
    chains=chains,
    warmup=warmup,
    sampling=sampling,
    adapt_delta=adapt_delta,
)

09:45:32 - cmdstanpy - INFO - compiling stan file /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/stan/logistic_response.stan to exe file /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/stan/logistic_response


Compiling logistic_response.stan


09:45:41 - cmdstanpy - INFO - compiled model executable: /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/stan/logistic_response


Running response


09:45:42 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/1500 [00:00<?, ?it/s, (Warmup)]


chain 1:   7%|▋         | 100/1500 [00:52<12:18,  1.89it/s, (Warmup)]



chain 1:  13%|█▎        | 200/1500 [01:57<12:55,  1.68it/s, (Warmup)]


chain 1:  20%|██        | 300/1500 [02:58<12:01,  1.66it/s, (Warmup)]


chain 1:  27%|██▋       | 400/1500 [03:44<10:00,  1.83it/s, (Warmup)]

chain 1:  33%|███▎      | 501/1500 [04:31<08:37,  1.93it/s, (Sampling)]








chain 1:  40%|████      | 600/1500 [05:25<07:54,  1.90it/s, (Sampling)]


chain 1:  47%|████▋     | 700/1500 [06:14<06:51,  1.94it/s, (Sampling)]


chain 1:  53%|█████▎    | 800/1500 [07:02<05:51,  1.99it/s, (Sampling)]


chain 1:  60%|██████    | 900/1500 [07:53<05:03,  1.98it/s, (Sampling)]


chain 1:  67%|██████▋   | 1000/1500 [08:48<04:19,  1.92it/s, (Sampling)]


chain 1:  73%|███████▎  | 1100/1500 [09:40<03:28,  1.92it/s, (Sampling)]

chain 1:  87%|████████▋ | 1300/1500 [11:23<01:43,  1.93it/s, (Sampling


10:00:07 - cmdstanpy - INFO - CmdStan done processing.



Saved idata to /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/response/idata.nc


## Variational Inference (Logistic Full)

In [ ]:
# Run variational inference (ADVI) for logistic_full
vi_outdir = out_root / 'full_vi'
vi_outdir.mkdir(parents=True, exist_ok=True)

if 'mdl_full' not in globals():
    mdl_full = compile_model(models['full']['file'])

fit_full_vi = mdl_full.variational(
    data=stan_data_full,
    seed=20260417,
    output_dir=str(vi_outdir),
)

print('Saved VI outputs to', vi_outdir)
print(fit_full_vi)

10:00:27 - cmdstanpy - INFO - Chain [1] start processing
10:00:39 - cmdstanpy - INFO - Chain [1] done processing


Saved VI outputs to /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/full_vi
CmdStanVB: model=logistic_full['method=variational', 'adapt', 'engaged=1']
 csv_file:
	/Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/full_vi/logistic_full-20260419100027.csv
 output_file:
	/Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/full_vi/logistic_full-20260419100027_0-stdout.txt
